# Exploring Tampa's published development records

A reproducible, public-facing introduction to the **Tampa Published Development Records (TDR)** dataset.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jaclenga/Tampa-Development-Records/blob/main/notebooks/tampa_development_exploration.ipynb)

**No installation is required.** Run all cells with a Python 3 Jupyter kernel. A repository checkout is read locally; hosted notebooks fall back to the same public files on GitHub.

## Read this before interpreting the charts

TDR archives records returned by eight named City of Tampa GIS layers. It is **not** a complete inventory of Tampa permits, construction, completed projects, or investment.

- `event_month` comes from a selected field in a City record; it is not the month TDR collected the record.
- Permit issuance, application creation, record creation, actual starts, and planned starts are not interchangeable events.
- A change between snapshots is a change in what a public layer reported. It does not by itself prove that physical work started, finished, or was cancelled.

See the [temporal cohort methodology](../docs/methodology/TEMPORAL_COHORTS.md), [known limitations](../docs/reference/KNOWN_LIMITATIONS.md), and [data dictionary](../docs/reference/data_dictionary.csv).

## 1. Setup

This notebook uses Python's standard library plus Jupyter's built-in display layer.

In [ ]:
from collections import Counter
from html import escape
from datetime import date
from pathlib import Path
from urllib.request import urlopen
import csv
import io
import json

from IPython.display import HTML, Markdown, display

REPOSITORY_URL = "https://github.com/Jaclenga/Tampa-Development-Records"
DATA_REVISION = "6959a5e7b038fa79024d350933d17e2b6c3bb477"
RAW_BASE = f"https://raw.githubusercontent.com/Jaclenga/Tampa-Development-Records/{DATA_REVISION}/"
MAX_REMOTE_BYTES = 10 * 1024 * 1024


def find_repository_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    return next(
        (p for p in candidates if (p / "data/monthly_events/index.json").is_file()),
        None,
    )


REPOSITORY_ROOT = find_repository_root()
DATA_MODE = (
    "local repository" if REPOSITORY_ROOT else f"GitHub revision {DATA_REVISION[:12]}"
)


def read_text(relative_path):
    relative_path = str(relative_path).replace("\\", "/").lstrip("/")
    if not relative_path or ".." in Path(relative_path).parts:
        raise ValueError(
            "Repository paths must be non-empty and cannot traverse parent directories."
        )
    if REPOSITORY_ROOT:
        return (REPOSITORY_ROOT / relative_path).read_text(encoding="utf-8-sig")
    with urlopen(RAW_BASE + relative_path, timeout=60) as response:  # noqa: S310 - fixed HTTPS origin
        content = response.read(MAX_REMOTE_BYTES + 1)
    if len(content) > MAX_REMOTE_BYTES:
        raise ValueError(
            f"Remote file exceeds the {MAX_REMOTE_BYTES:,}-byte safety limit: {relative_path}"
        )
    return content.decode("utf-8-sig")


def read_json(path):
    return json.loads(read_text(path))


def read_csv(path):
    return list(csv.DictReader(io.StringIO(read_text(path))))


def label(value):
    return str(value).replace("_", " ").title()


def show_table(rows, columns):
    head = "".join(f"<th>{escape(title)}</th>" for _, title in columns)
    body = []
    for row in rows:
        values = []
        for key, _ in columns:
            value = row.get(key, "")
            value = f"{value:,}" if isinstance(value, int) else value
            values.append(f"<td>{escape(str(value))}</td>")
        body.append("<tr>" + "".join(values) + "</tr>")
    css = """<style>.tdr{border-collapse:collapse;width:100%;font:14px system-ui}.tdr th{background:#17324d;color:white;text-align:left}.tdr th,.tdr td{padding:7px 9px;border:1px solid #d7dee5}.tdr tr:nth-child(even){background:#f5f8fa}</style>"""
    display(
        HTML(
            css
            + f"<table class='tdr'><thead><tr>{head}</tr></thead><tbody>{''.join(body)}</tbody></table>"
        )
    )


def show_cards(items):
    cards = []
    for title, value, note in items:
        cards.append(
            "<div style='border:1px solid #ccd7df;border-top:4px solid #e07a32;border-radius:6px;padding:12px'>"
            + f"<small>{escape(title)}</small><div style='font:700 26px system-ui;color:#17324d'>{escape(str(value))}</div>"
            + f"<small>{escape(note)}</small></div>"
        )
    display(
        HTML(
            "<div style='display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));gap:10px'>"
            + "".join(cards)
            + "</div>"
        )
    )


def line_chart(items, title, color="#087e8b", width=900, height=310):
    if not items:
        display(Markdown("*No rows match this selection.*"))
        return
    left, right, top, bottom = 55, 18, 40, 50
    plot_w, plot_h = width - left - right, height - top - bottom
    maximum = max(v for _, v in items) or 1
    points = [
        (left + plot_w * i / max(1, len(items) - 1), top + plot_h * (1 - v / maximum))
        for i, (_, v) in enumerate(items)
    ]
    svg = [
        f"<svg role='img' aria-label='{escape(title)}' viewBox='0 0 {width} {height}' style='max-width:100%;height:auto;font-family:system-ui'>",
        f"<text x='{left}' y='22' font-size='17' font-weight='700' fill='#17324d'>{escape(title)}</text>",
    ]
    for fraction in (0, 0.25, 0.5, 0.75, 1):
        y, value = top + plot_h * (1 - fraction), round(maximum * fraction)
        svg += [
            f"<line x1='{left}' y1='{y:.1f}' x2='{width - right}' y2='{y:.1f}' stroke='#dce4e9'/>",
            f"<text x='{left - 7}' y='{y + 4:.1f}' text-anchor='end' font-size='10'>{value:,}</text>",
        ]
    svg.append(
        "<polyline fill='none' stroke='{}' stroke-width='3' points='{}'/>".format(
            color, " ".join(f"{x:.1f},{y:.1f}" for x, y in points)
        )
    )
    ticks = sorted(
        set(
            [0, len(items) - 1]
            + [i for i, (name, _) in enumerate(items) if name.endswith("-01")]
        )
    )
    for i in ticks:
        svg.append(
            f"<text x='{points[i][0]:.1f}' y='{height - 20}' text-anchor='middle' font-size='10'>{escape(items[i][0])}</text>"
        )
    display(HTML("".join(svg) + "</svg>"))


def bar_chart(items, title, color="#e07a32", width=900):
    if not items:
        display(Markdown("*No rows match this selection.*"))
        return
    display_names = [
        str(name) if len(str(name)) <= 56 else str(name)[:53] + "..."
        for name, _ in items
    ]
    longest_label = max(len(name) for name in display_names)
    row_h, right, top = 27, 70, 40
    left = min(int(width * 0.48), max(180, int(longest_label * 6.6) + 16))
    height, maximum = top + row_h * len(items) + 15, max(v for _, v in items) or 1
    svg = [
        f"<svg role='img' aria-label='{escape(title)}' viewBox='0 0 {width} {height}' style='max-width:100%;height:auto;font-family:system-ui'>",
        f"<text x='0' y='22' font-size='17' font-weight='700' fill='#17324d'>{escape(title)}</text>",
    ]
    for i, ((name, value), display_name) in enumerate(zip(items, display_names)):
        y, bar_w = top + i * row_h, (width - left - right) * value / maximum
        svg += [
            f"<text x='{left - 8}' y='{y + 15}' text-anchor='end' font-size='11'><title>{escape(str(name))}</title>{escape(display_name)}</text>",
            f"<rect x='{left}' y='{y + 3}' width='{bar_w:.1f}' height='17' rx='2' fill='{color}'/>",
            f"<text x='{left + bar_w + 7:.1f}' y='{y + 16}' font-size='11'>{value:,}</text>",
        ]
    display(HTML("".join(svg) + "</svg>"))


print(f"Data source: {DATA_MODE}")

## 2. Load the canonical cohort

The selection below reproduces the published `monthly_events` boundary: the source date must be on or after January 1, 2020 and on or before the snapshot supplying the row. Explicit future plans remain separate.

In [ ]:
release = read_json("manifest.json")
monthly_index = read_json("data/monthly_events/index.json")
planned_index = read_json("data/planned_events/index.json")
change_index = read_json("data/monthly_changes/index.json")
activity_rows = read_csv("data/processed/activity_by_month.csv")

dataset_start = monthly_index["dataset_start_date"]
events = [
    row
    for row in activity_rows
    if row["event_date"] and dataset_start <= row["event_date"] <= row["snapshot_date"]
]

# Fail loudly if a checkout is stale or internally inconsistent, even under python -O.
if len(events) != monthly_index["record_count"]:
    raise RuntimeError(
        "Canonical cohort does not reconcile with the monthly-events index."
    )
if len({row["record_id"] for row in activity_rows}) != len(activity_rows):
    raise RuntimeError("Canonical cohort contains duplicate record IDs.")
if any(row["event_date_is_after_snapshot"] != "0" for row in events):
    raise RuntimeError("Non-future event selection contains a date after its snapshot.")

latest_snapshot_item = max(
    change_index["snapshots"], key=lambda item: item["snapshot_date"]
)
latest_snapshot = latest_snapshot_item["snapshot_date"]
show_cards(
    [
        ("Release", release["version"], "Repository manifest"),
        (
            "Canonical identities",
            f"{len(activity_rows):,}",
            "Dated, planned, and undated",
        ),
        ("Non-future events", f"{len(events):,}", f"Since {dataset_start}"),
        (
            "Event months",
            monthly_index["month_count"],
            f"{monthly_index['first_event_month']} to {monthly_index['last_event_month']}",
        ),
        ("Future plans", f"{planned_index['record_count']:,}", "Kept separate"),
        (
            "Latest snapshot",
            latest_snapshot,
            f"{latest_snapshot_item['record_count']:,} source records",
        ),
    ]
)

## 3. Source-described dates over time

This overview shows the archive's shape, but it is **not a single development-activity trend**: it mixes sources and date meanings. The latest event month may also be incomplete.

In [ ]:
monthly_totals = [
    (item["event_month"], item["record_count"]) for item in monthly_index["months"]
]
line_chart(monthly_totals, "All retained source-described dates (mixed meanings)")
display(
    Markdown(
        f"**Coverage note:** the latest snapshot is `{latest_snapshot}`; the `{monthly_index['last_event_month']}` point is not a complete calendar month."
    )
)

## 4. Source and date meanings

A defensible comparison holds both `source_name` and `event_date_type` constant.

In [ ]:
pair_counts = Counter((row["source_name"], row["event_date_type"]) for row in events)
pair_rows = [
    {
        "source": label(source),
        "date_type": label(date_type),
        "records": count,
        "share": f"{count / len(events):.1%}",
    }
    for (source, date_type), count in pair_counts.most_common()
]
show_table(
    pair_rows,
    [
        ("source", "City source layer"),
        ("date_type", "Selected date meaning"),
        ("records", "Records"),
        ("share", "Share"),
    ],
)

## 5. A meaning-preserving trend

Edit `SOURCE` and `EVENT_TYPE` using a pairing from the table above. The default counts selected permit-issued dates from the Single-Family Permits layer—not housing units built or construction starts.

In [ ]:
SOURCE = "single_family_permits"
EVENT_TYPE = "permit_issued"

if (SOURCE, EVENT_TYPE) not in pair_counts:
    raise ValueError("Choose a source/date-type pairing shown in section 4.")
selected_rows = [
    row
    for row in events
    if row["source_name"] == SOURCE and row["event_date_type"] == EVENT_TYPE
]
selected_counts = Counter(row["event_month"] for row in selected_rows)
selected_series = [
    (month, selected_counts.get(month, 0)) for month, _ in monthly_totals
]
line_chart(selected_series, f"{label(SOURCE)} — {label(EVENT_TYPE)}", color="#e07a32")

annual = Counter()
for month, count in selected_series:
    annual[month[:4]] += count
annual_rows = []
for year, count in sorted(annual.items()):
    coverage = (
        f"Partial through snapshot {latest_snapshot}"
        if year == latest_snapshot[:4]
        else "Calendar year as retained"
    )
    annual_rows.append({"year": year, "records": count, "coverage": coverage})
show_table(
    annual_rows,
    [
        ("year", "Event year"),
        ("records", "Selected records"),
        ("coverage", "Coverage note"),
    ],
)

## 6. Neighborhood profile without street-level detail

Although source files contain public site addresses, this exploration stays aggregate. Neighborhood groups with fewer than five selected records are omitted. Labels are source-reported, not spatially recomputed.

In [ ]:
MINIMUM_GROUP_SIZE = 5
neighborhood_counts = Counter(
    row["neighborhood"].strip() for row in selected_rows if row["neighborhood"].strip()
)
public_neighborhoods = [
    (name, count)
    for name, count in neighborhood_counts.most_common()
    if count >= MINIMUM_GROUP_SIZE
]
bar_chart(
    public_neighborhoods[:15],
    f"Top neighborhoods: {label(SOURCE)} / {label(EVENT_TYPE)}",
)
blank_neighborhood = sum(not row["neighborhood"].strip() for row in selected_rows)
suppressed_groups = sum(
    count < MINIMUM_GROUP_SIZE for count in neighborhood_counts.values()
)
display(
    Markdown(
        f"Of **{len(selected_rows):,}** selected records, **{blank_neighborhood:,}** have no neighborhood label; **{suppressed_groups:,}** small groups are omitted."
    )
)

## 7. Completeness profile

Blankness is not the same as error. Source layers publish different fields, so profile missingness within a source before using a field analytically.

In [ ]:
profile = []
for field in [
    "activity_id",
    "status",
    "project_name",
    "record_type",
    "address",
    "neighborhood",
]:
    blank = sum(not row[field].strip() for row in events)
    profile.append(
        {
            "field": field,
            "populated": len(events) - blank,
            "blank": blank,
            "complete": f"{(len(events) - blank) / len(events):.1%}",
        }
    )
show_table(
    profile,
    [
        ("field", "Field"),
        ("populated", "Populated"),
        ("blank", "Blank"),
        ("complete", "Completeness"),
    ],
)

## 8. What changed between archived snapshots?

The code selects the most recent archived comparison and reports its exact duration and analytical flags. Treat it as a publication-layer comparison unless the metadata explicitly identifies it as canonical and usable for aggregate trends.

In [ ]:
comparison = max(
    change_index["comparisons"], key=lambda item: item["after_snapshot_date"]
)
changes = read_csv(comparison["csv"])
interval_days = (
    date.fromisoformat(comparison["after_snapshot_date"])
    - date.fromisoformat(comparison["before_snapshot_date"])
).days
change_types = Counter(row["change_type"] for row in changes)
source_changes = Counter(row["source_name"] for row in changes)

show_cards(
    [
        (
            "Interval",
            f"{comparison['before_snapshot_date']} → {comparison['after_snapshot_date']}",
            f"{interval_days}-day comparison",
        ),
        (
            "Changed records",
            f"{comparison['records_with_any_published_change']:,}",
            "At least one published-field change",
        ),
        ("Change rows", f"{len(changes):,}", "A record can contribute multiple rows"),
        (
            "Warnings",
            comparison["warning_alert_count"],
            f"Status: {comparison['analysis_status']}",
        ),
    ]
)
comparison_guidance = (
    "This comparison is canonical and marked usable for aggregate trends."
    if comparison["canonical_monthly_comparison"]
    and comparison["usable_for_global_aggregate_trend"]
    else "This comparison is not approved as a canonical global aggregate trend; interpret source-level publication changes only."
)
display(Markdown(f"**Interpretation guardrail:** {comparison_guidance}"))
bar_chart(
    [(label(name), count) for name, count in change_types.most_common(12)],
    "Most frequent published change types",
    color="#087e8b",
)
show_table(
    [
        {"source": label(name), "changes": count}
        for name, count in source_changes.most_common()
    ],
    [("source", "City source layer"), ("changes", "Change rows")],
)

## 9. Reproducible takeaways

This notebook is a starting point, not a causal study. For publication, report the release version, snapshot dates, exact filters, and source/date-type pairing. Cite the dataset using [`CITATION.cff`](../CITATION.cff), review [`DATA_LICENSE.md`](../DATA_LICENSE.md), and do not claim a measured error rate while manual validation remains incomplete.

Good next questions include year-over-year patterns within one stable source/date pair, neighborhood differences after accounting for missing labels, and which snapshot changes survive manual review.

In [ ]:
analysis_receipt = {
    "repository": REPOSITORY_URL,
    "release_version": release["version"],
    "remote_data_revision": DATA_REVISION,
    "data_mode": DATA_MODE,
    "dataset_start_date": dataset_start,
    "latest_snapshot_date": latest_snapshot,
    "non_future_event_records": len(events),
    "selected_source": SOURCE,
    "selected_event_date_type": EVENT_TYPE,
    "selected_records": len(selected_rows),
    "minimum_public_group_size": MINIMUM_GROUP_SIZE,
    "comparison_is_canonical_monthly": comparison["canonical_monthly_comparison"],
    "comparison_usable_for_global_trend": comparison[
        "usable_for_global_aggregate_trend"
    ],
}
print(json.dumps(analysis_receipt, indent=2))